**07 - Aggregate Scores**

Aggregates all 5 pillar scores (transparency, fairness, robustness, explainability, privacy) into a final trustworthiness index.

In [1]:
# Importing necessary libraries
import json
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [6]:
def load(path, key):
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = data[key]
    return {r['model_id']: r for r in data}
transparency   = load('../results/transparency_scores.json','transparency')
fairness       = load('../results/fairness_scores.json','fairness')
robustness     = load('../results/robustness_scores.json','robustness')
explainability = load('../results/explainability_scores.json','explainability')
privacy        = load('../results/privacy_scores.json','privacy')
models = list(transparency.keys())
print(f'loaded scores for {len(models)} models: {models}')

loaded scores for 5 models: ['TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'microsoft/phi-1_5', 'Qwen/Qwen2-0.5B', 'HuggingFaceTB/SmolLM-360M', 'stabilityai/stablelm-2-1_6b']


In [8]:
weights = {
    'transparency':   0.15,
    'fairness':       0.25,
    'robustness':     0.25,
    'explainability': 0.20,
    'privacy':        0.15,
}
records = []
for model_id in models:
    t = t = transparency[model_id]['transparency_score']
    f = fairness[model_id]['fairness_score']
    r = robustness[model_id]['robustness_score']
    e = explainability[model_id]['explainability_score']
    p = privacy[model_id]['privacy_score']
    trust = round(
        weights['transparency']   * t +
        weights['fairness']       * f +
        weights['robustness']     * r +
        weights['explainability'] * e +
        weights['privacy']        * p,
        4
    )

    records.append({
        'model_id':        model_id,
        'transparency':    t,
        'fairness':        f,
        'robustness':      r,
        'explainability':  e,
        'privacy':         p,
        'trustworthiness': trust,
    })
df = pd.DataFrame(records).sort_values('trustworthiness', ascending=False).reset_index(drop=True)
df.index += 1
print(df[['model_id', 'trustworthiness']].to_string())

                             model_id  trustworthiness
1         stabilityai/stablelm-2-1_6b           0.5652
2           HuggingFaceTB/SmolLM-360M           0.5519
3                   microsoft/phi-1_5           0.5330
4  TinyLlama/TinyLlama-1.1B-Chat-v1.0           0.5072
5                     Qwen/Qwen2-0.5B           0.4791


In [9]:
os.makedirs('../results/', exist_ok=True)
output = {
    'weights': weights,
    'models':  df.to_dict(orient='records'),
}
with open('../results/aggregate_scores.json', 'w') as f:
    json.dump(output, f, indent=2)
print('saved to outputs/aggregate_scores.json')
print(df[['model_id', 'trustworthiness']].to_string())

saved to outputs/aggregate_scores.json
                             model_id  trustworthiness
1         stabilityai/stablelm-2-1_6b           0.5652
2           HuggingFaceTB/SmolLM-360M           0.5519
3                   microsoft/phi-1_5           0.5330
4  TinyLlama/TinyLlama-1.1B-Chat-v1.0           0.5072
5                     Qwen/Qwen2-0.5B           0.4791
